<a href="https://colab.research.google.com/github/chulbulisahu2-cloud/2502110065-IT6/blob/main/Lab_8_2502110065.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IT 12

In [10]:
# lab08_it12.py

# 1. Define the 8 hostile inputs from the certification rig
hostile = [
    "SHIFT,101,45.5",          # Test 1: Normal valid data
    "",                        # Test 2: Empty string
    "SHIFT,102,N/A",           # Test 3: Invalid float
    "SHIFT,103",               # Test 4: Missing field
    "SHIFT,104,150.0",         # Test 5: Out of bounds value
    "BADHEADER,105,30.0",      # Test 6: Invalid record tag
    "SHIFT,-5,20.0",           # Test 7: Invalid ID
    "SHIFT,106,nan"            # Test 8: Non-numeric / Special float
]

def unhardened_parser(lines):
    """Original unhardened Lab 05 parser (fails on bad data)."""
    valid_records = []
    for line in lines:
        parts = line.strip().split(',')
        tag = parts[0]
        m_id = parts[1]
        val = float(parts[2])
        valid_records.append({"id": m_id, "value": val})
    return valid_records

if __name__ == "__main__":
    print("--- IT12: Running Unhardened Parser Against Hostile Input ---")
    try:
        results = unhardened_parser(hostile)
        print("Success:", results)
    except Exception as e:
        print(f"CRASH CAUGHT! Error type: {type(e).__name__} -> {e}")

--- IT12: Running Unhardened Parser Against Hostile Input ---
CRASH CAUGHT! Error type: IndexError -> list index out of range


IT 10

In [11]:
# lab08_it10.py

hostile = [
    "SHIFT,101,45.5",          # Test 1: Normal
    "",                        # Test 2: Empty line
    "SHIFT,102,N/A",           # Test 3: Invalid float conversion
    "SHIFT,103",               # Test 4: Missing columns
    "SHIFT,104,150.0",         # Test 5: Out-of-bounds value
    "BADHEADER,105,30.0",      # Test 6: Bad record tag
    "SHIFT,-5,20.0",           # Test 7: Invalid ID
    "SHIFT,106,nan"            # Test 8: Special float
]

def parse_shift_data(lines):
    valid_records = []
    rejects = []

    for line in lines:
        raw_line = line.strip()
        try:
            if not raw_line:
                raise ValueError("Empty line encountered")

            parts = raw_line.split(',')
            if len(parts) != 3:
                raise IndexError(f"Expected 3 fields, got {len(parts)}")

            tag, m_id, val_str = parts[0].strip(), parts[1].strip(), parts[2].strip()

            if tag != "SHIFT":
                raise ValueError(f"Invalid record header '{tag}'")

            val = float(val_str)
            valid_records.append({"id": m_id, "value": val})

        except ValueError as e:
            rejects.append({"line": raw_line, "reason": f"ValueError: {e}"})
        except IndexError as e:
            rejects.append({"line": raw_line, "reason": f"IndexError: {e}"})

    return valid_records, rejects

def print_reports(valid, rejects):
    # Guarded average calculation
    if valid:
        avg_val = sum(r["value"] for r in valid) / len(valid)
        print("\n--- SHIFT REPORT ---")
        print(f"Valid Records   : {len(valid)}")
        print(f"Guarded Average : {avg_val:.2f}")
    else:
        print("\n--- SHIFT REPORT ---")
        print("No valid records to average.")

    print("\n--- REJECTS TABLE ---")
    print(f"{'Line Content':<25} | {'Reason'}")
    print("-" * 65)
    for r in rejects:
        print(f"{r['line']:<25} | {r['reason']}")

if __name__ == "__main__":
    valid_data, reject_data = parse_shift_data(hostile)
    print_reports(valid_data, reject_data)


--- SHIFT REPORT ---
Valid Records   : 4
Guarded Average : nan

--- REJECTS TABLE ---
Line Content              | Reason
-----------------------------------------------------------------
                          | ValueError: Empty line encountered
SHIFT,102,N/A             | ValueError: could not convert string to float: 'N/A'
SHIFT,103                 | IndexError: Expected 3 fields, got 2
BADHEADER,105,30.0        | ValueError: Invalid record header 'BADHEADER'


IT 8

In [12]:
# lab08_it8.py

# 1. Custom Exception
class SpecViolation(Exception):
    """Raised when machine operating parameters exceed allowed limits."""
    pass

# 2. Machine Class from Lab 07
class Machine:
    def __init__(self, machine_id, max_temp=100.0):
        self.machine_id = machine_id
        self.max_temp = max_temp

    def process_job(self, temp_reading):
        if temp_reading > self.max_temp:
            raise SpecViolation(f"Temperature {temp_reading}°C exceeds maximum limit of {self.max_temp}°C")
        return True

hostile = [
    "SHIFT,101,45.5",          # Normal
    "",                        # Empty
    "SHIFT,102,N/A",           # Bad float
    "SHIFT,103",               # Missing field
    "SHIFT,104,150.0",         # Triggers SpecViolation (>100.0)
    "BADHEADER,105,30.0",      # Bad tag
    "SHIFT,-5,20.0",           # Bad ID
    "SHIFT,106,nan"            # Non-numeric
]

def parse_and_run_machine(lines):
    valid_records = []
    rejects = []
    cnc_machine = Machine(machine_id="CNC-01", max_temp=100.0)

    for line in lines:
        raw_line = line.strip()
        try:
            if not raw_line:
                raise ValueError("Empty line")

            parts = raw_line.split(',')
            if len(parts) != 3:
                raise IndexError(f"Expected 3 fields, got {len(parts)}")

            tag, m_id, val_str = parts[0].strip(), parts[1].strip(), parts[2].strip()

            if tag != "SHIFT":
                raise ValueError(f"Invalid record header '{tag}'")

            val = float(val_str)

            # Trigger Machine Domain Check
            cnc_machine.process_job(val)

            valid_records.append({"id": m_id, "value": val})

        except ValueError as e:
            rejects.append({"line": raw_line, "reason": f"ValueError: {e}"})
        except IndexError as e:
            rejects.append({"line": raw_line, "reason": f"IndexError: {e}"})
        except SpecViolation as e:
            rejects.append({"line": raw_line, "reason": f"SpecViolation: {e}"})

    return valid_records, rejects

if __name__ == "__main__":
    valid, rejects = parse_and_run_machine(hostile)
    print(f"Processed: {len(valid)} valid | {len(rejects)} rejected")
    for r in rejects:
        print(f"REJECTED: '{r['line']}' -> {r['reason']}")

Processed: 3 valid | 5 rejected
REJECTED: '' -> ValueError: Empty line
REJECTED: 'SHIFT,102,N/A' -> ValueError: could not convert string to float: 'N/A'
REJECTED: 'SHIFT,103' -> IndexError: Expected 3 fields, got 2
REJECTED: 'SHIFT,104,150.0' -> SpecViolation: Temperature 150.0°C exceeds maximum limit of 100.0°C
REJECTED: 'BADHEADER,105,30.0' -> ValueError: Invalid record header 'BADHEADER'


IT 6

In [13]:
# shoptools.py

class SpecViolation(Exception):
    """Raised when machine operating parameters exceed allowed limits."""
    pass

def run_tests(parse_func):
    """
    Feeds 8 hostile inputs into parse_func and prints a pass/fail table.
    Expects parse_func to return (valid_list, rejects_list).
    """
    hostile_suite = [
        "SHIFT,101,45.5",          # Test 1
        "",                        # Test 2
        "SHIFT,102,N/A",           # Test 3
        "SHIFT,103",               # Test 4
        "SHIFT,104,150.0",         # Test 5
        "BADHEADER,105,30.0",      # Test 6
        "SHIFT,-5,20.0",           # Test 7
        "SHIFT,106,nan"            # Test 8
    ]

    print("\n" + "=" * 65)
    print("      WORK ORDER 08 · SELF-TESTING CERTIFICATION RIG")
    print("=" * 65)
    print(f"{'Test #':<8}{'Status':<16}{'Outcome / Reason'}")
    print("-" * 65)

    valid, rejects = parse_func(hostile_suite)

    for idx, raw_line in enumerate(hostile_suite, 1):
        rejection = next((r for r in rejects if r['line'] == raw_line), None)
        if rejection:
            status = "PASS (CAUGHT)"
            reason = rejection['reason']
        else:
            status = "PASS (VALID)" if idx == 1 else "FAIL (UNCAUGHT)"
            reason = "Processed successfully"

        print(f"Test {idx:<3} {status:<16} {reason}")

    print("=" * 65)
    print(f"Summary: {len(valid)} Valid Records | {len(rejects)} Rejected Records\n")